# Arquivos binários: `fread`, `fwrite` e `fseek` — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Na aula passada, `turma.csv` era lido linha a linha: para chegar ao aluno 3.000, liam-se as 2.999 linhas
anteriores, e trocar uma nota exigia reescrever o arquivo inteiro. Este tutorial segue o caminho dos
slides de hoje: gravar a `struct aluno` no disco exatamente como ela está na memória, ler de volta, saltar
direto ao registro *k* e alterá-lo no lugar. Depois de cada bloco há exercícios com teste automático; no
fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Explicar a diferença entre gravar um número como texto (`fprintf`) e como bytes (`fwrite`)
- Gravar e ler registros e vetores com `fwrite` e `fread`, testando o número de elementos devolvido
- Usar `sizeof` para o tamanho de um registro e explicar o preenchimento e a ordem dos bytes no arquivo
- Explicar por que uma `struct` com ponteiros não pode ir para o disco com `fwrite`
- Ir direto ao registro *k* com `fseek`, medir um arquivo com `ftell` e alterar registros com `"r+b"`


In [ ]:

# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: algum laço não termina?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip('\n').split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O comando terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
    if r.stderr:
        print('stderr:', r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)


Os exercícios usam alguns arquivos binários de teste. A célula abaixo os cria com o módulo `struct` do
Python, no mesmo formato da `struct aluno` — você vai entender o `'=i50s2x4d'` depois da seção 2.


In [ ]:
import random, struct

FMT = '=i50s2x4d'          # int, char[50], 2 bytes de preenchimento, double[4]

def registro(mat, nome, v):
    return struct.pack(FMT, mat, nome.encode(), *v)

turma = [(202600101, 'Ana Souza', (8.0, 6.5, 9.0, 7.5)),
         (202600102, 'Bruno Lima', (5.0, 7.0, 4.0, 7.0)),
         (202600103, 'Carla Menezes Santos', (9.0, 10.0, 8.5, 9.5)),
         (202600104, 'Davi Rocha', (3.5, 4.0, 6.0, 2.5))]
bin_turma = b''.join(registro(*t) for t in turma)
open('turma_ex.bin', 'wb').write(bin_turma)
open('corrompido.bin', 'wb').write(bin_turma + bytes(10))

random.seed(512)
img = bytes(random.randrange(256) for _ in range(10000))
open('img.bin', 'wb').write(img)
img2 = bytearray(img)
img2[41] ^= 0xff
open('img2.bin', 'wb').write(img2)
open('pref.bin', 'wb').write(img[:100])
open('vazio.bin', 'wb').close()
open('ola.bin', 'wb').write(b'Ola, mundo!\n\x00\x01\xff\x7f')
open('curto.bin', 'wb').write(b'0123456789abcdefABCD')

mats = sorted(random.sample(range(202600000, 202700000), 1000))
grande = [(m, f'Aluno {i:04d}', (5.0, 5.0, 5.0, 5.0)) for i, m in enumerate(mats)]
open('grande.bin', 'wb').write(b''.join(registro(*t) for t in grande))
print('arquivos de teste criados')


## 1. Texto ou bytes?

`fprintf` **converte** o número em caracteres; `fwrite` **copia** os bytes que estão na memória. O
programa abaixo grava o mesmo `double` dos dois jeitos e lê de volta. Antes de executar, responda: os
dois valores lidos são iguais a `x`? Quantos bytes tem cada arquivo?


In [ ]:
%%writefile precisao.c
/* precisao.c --- o mesmo double gravado como texto e como bytes */
#include <stdio.h>

int main(void)
{
    double x = 1.0 / 3.0, lido_t, lido_b;

    FILE *ft = fopen("x.txt", "w");
    FILE *fb = fopen("x.bin", "wb");
    if (ft == NULL || fb == NULL) {
        perror("x.txt/x.bin");
        return 1;
    }
    fprintf(ft, "%f", x);              /* texto: caracteres */
    fwrite(&x, sizeof x, 1, fb);       /* binario: os bytes de x */
    fclose(ft);
    fclose(fb);

    ft = fopen("x.txt", "r");
    fb = fopen("x.bin", "rb");
    if (ft == NULL || fb == NULL) {
        perror("x.txt/x.bin");
        return 1;
    }
    fscanf(ft, "%lf", &lido_t);
    fread(&lido_b, sizeof lido_b, 1, fb);
    fclose(ft);
    fclose(fb);

    printf("texto:   %.17f  %s\n", lido_t, lido_t == x ? "igual" : "diferente");
    printf("binario: %.17f  %s\n", lido_b, lido_b == x ? "igual" : "diferente");
    return 0;
}


In [ ]:

!gcc -Wall -Wextra precisao.c -o precisao && ./precisao
!wc -c x.txt x.bin
!xxd x.bin


`%f` escreve seis casas: o texto arredondou, e o `fscanf` não tem como recuperar o que foi jogado fora.
O binário guardou os 8 bytes de `x` sem conversão. Experimente trocar `"%f"` por `"%.17g"`: o texto
passa a ser exato, mas ocupa 19 bytes.


## 2. Gravar bytes: `fwrite`

```c
size_t fwrite(const void *p, size_t tam, size_t n, FILE *f);
```

`fwrite` grava `n` elementos de `tam` bytes cada, a partir do endereço `p`, e devolve quantos
**elementos** gravou. O modo de abertura leva um `b`: `"wb"` (apaga e grava), `"rb"`, `"ab"`, `"r+b"`.
No Linux e no macOS o `b` não muda nada; no Windows, sem ele, cada `'\n'` gravado vira dois bytes.

O programa da aula converte `turma.csv`, da aula passada, em `turma.bin`: um registro `struct aluno`
por aluno, um atrás do outro.


In [ ]:
%%writefile turma.csv
202600101;Ana Souza;8.0;6.5;9.0;7.5
202600102;Bruno Lima;5.0;7.0;4.0;7.0
202600103;Carla Menezes Santos;9.0;10.0;8.5;9.5
202600104;Davi Rocha;3.5;4.0;6.0;2.5


In [ ]:
%%writefile aluno.h
/* aluno.h --- o registro gravado em turma.bin */
#ifndef ALUNO_H
#define ALUNO_H

#define NAV 4   /* avaliacoes por aluno */

struct aluno {
    int    mat;
    char   nome[50];
    double v[NAV];
};

#endif


In [ ]:
%%writefile cadastro.c
/* cadastro.c --- converte turma.csv (texto) em turma.bin (registros)
   uso: ./cadastro                        (turma.csv -> turma.bin)
        ./cadastro entrada.csv saida.bin */
#include <stdio.h>
#include <string.h>
#include "aluno.h"

int main(int argc, char *argv[])
{
    const char *nome_ent = argc > 1 ? argv[1] : "turma.csv";
    const char *nome_sai = argc > 2 ? argv[2] : "turma.bin";

    FILE *ent = fopen(nome_ent, "r");
    if (ent == NULL) {
        perror(nome_ent);
        return 1;
    }
    FILE *sai = fopen(nome_sai, "wb");
    if (sai == NULL) {
        perror(nome_sai);
        fclose(ent);
        return 1;
    }

    char linha[256];
    int n = 0;
    while (fgets(linha, sizeof linha, ent) != NULL) {
        struct aluno a = {0};          /* comeca zerado: sem lixo no disco */
        if (sscanf(linha, "%d;%49[^;];%lf;%lf;%lf;%lf", &a.mat, a.nome,
                   &a.v[0], &a.v[1], &a.v[2], &a.v[3]) != 6)
            continue;
        if (fwrite(&a, sizeof a, 1, sai) != 1) {
            perror(nome_sai);
            break;
        }
        n++;
    }
    fclose(ent);
    fclose(sai);

    printf("%d alunos gravados em %s\n", n, nome_sai);
    return 0;
}


Antes de executar: um `int` tem 4 bytes e um `double`, 8. Quantos bytes terá `turma.bin`?


In [ ]:

!gcc -Wall -Wextra cadastro.c -o cadastro && ./cadastro
!wc -c turma.bin
!strings turma.bin
!xxd -l 88 turma.bin


A conta $4 + 50 + 4 \times 8 = 86$ daria 344 bytes, mas o arquivo tem 352. O `double` precisa começar
num endereço múltiplo de 8, e o compilador põe 2 bytes de **preenchimento** entre `nome` (que termina no
byte 54) e `v` (que começa no 56). O tamanho de um registro é sempre `sizeof(struct aluno)`, nunca a soma
feita à mão. Confira com `offsetof`, que dá a posição de cada campo:


In [ ]:
%%writefile layout.c
#include <stdio.h>
#include <stddef.h>
#include "aluno.h"

int main(void)
{
    printf("sizeof(struct aluno) = %zu\n", sizeof(struct aluno));
    printf("mat  comeca em %2zu\n", offsetof(struct aluno, mat));
    printf("nome comeca em %2zu\n", offsetof(struct aluno, nome));
    printf("v    comeca em %2zu\n", offsetof(struct aluno, v));
    return 0;
}


In [ ]:
!gcc -Wall -Wextra layout.c -o layout && ./layout


Repare também nos primeiros bytes do `xxd`: `a56e 130c`. A matrícula 202600101 é `0x0c136ea5`, gravada de
trás para frente — o byte menos significativo primeiro (*little endian*). Ordem dos bytes, tamanho dos
tipos e preenchimento dependem da máquina: um arquivo binário gravado numa arquitetura pode não ser
lido corretamente em outra.

### Exercício 1 — quadrados em disco

`./quadrados n saida.bin` grava os `int` $1, 4, 9, \ldots, n^2$ com **um único** `fwrite`: aloque o
vetor com `malloc`, preencha-o e grave tudo de uma vez. Se o arquivo não puder ser criado, imprima o
motivo com `perror` e termine com status 1. O teste lê o arquivo com Python e mostra o tamanho e os
valores.


In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 1: ./quadrados n saida.bin
   TODO: aloque um vetor de n int, preencha com i*i (i de 1 a n) e grave
   tudo com UM fwrite. Libere o vetor e feche o arquivo. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s n saida.bin\n", argv[0]);
        return 1;
    }
    int n = atoi(argv[1]);
    const char *nome = argv[2];

    /* TODO: implemente aqui */
    (void) n;
    (void) nome;

    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex1.c -o ex1
print('n = 5' + ':', end=' ')
testa('rm -f q.bin; ./ex1 5 q.bin && python3 -c "import struct; d = open(\'q.bin\', \'rb\').read(); print(len(d), list(struct.unpack(\'=%di\' % (len(d) // 4), d)))"', '20 [1, 4, 9, 16, 25]')
print('n = 1000' + ':', end=' ')
testa('rm -f q.bin; ./ex1 1000 q.bin && python3 -c "import struct; d = open(\'q.bin\', \'rb\').read(); v = struct.unpack(\'=%di\' % (len(d) // 4), d); print(len(d), v[0], v[499], v[-1])"', '4000 1 250000 1000000')
print('diretório que não existe' + ':', end=' ')
testa('./ex1 3 /nao/existe/q.bin; echo "status $?"', 'status 1')


## 3. De volta: `fread`

```c
size_t fread(void *p, size_t tam, size_t n, FILE *f);
```

`fread` lê até `n` elementos de `tam` bytes para o endereço `p` e devolve quantos conseguiu — no fim do
arquivo, menos que `n` (ou 0). O laço testa esse valor, como o `fscanf(...) == 5` da aula passada:


In [ ]:
%%writefile lista.c
/* lista.c --- le turma.bin registro a registro e imprime o boletim */
#include <stdio.h>
#include "aluno.h"

int main(int argc, char *argv[])
{
    const char *nome = argc > 1 ? argv[1] : "turma.bin";
    FILE *f = fopen(nome, "rb");
    if (f == NULL) {
        perror(nome);
        return 1;
    }
    struct aluno a;
    while (fread(&a, sizeof a, 1, f) == 1) {
        double m = (a.v[0] + a.v[1] + a.v[2] + a.v[3]) / NAV;
        printf("%d %-22s %5.2f\n", a.mat, a.nome, m);
    }
    fclose(f);
    return 0;
}


In [ ]:
!gcc -Wall -Wextra lista.c -o lista && ./lista


Agora uma `struct` com um **ponteiro** dentro. Antes de executar: quantos bytes `sizeof c` informa? A
segunda execução consegue imprimir o nome?


In [ ]:
%%writefile ponteiro.c
/* ponteiro.c --- grava uma struct que tem um ponteiro dentro
   uso: ./ponteiro grava    depois    ./ponteiro le */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

struct contato {
    int   mat;
    char *nome;          /* ponteiro, nao vetor */
};

int main(int argc, char *argv[])
{
    struct contato c;
    if (argc > 1 && strcmp(argv[1], "grava") == 0) {
        c.mat = 202600101;
        c.nome = malloc(50);
        strcpy(c.nome, "Ana Souza");
        FILE *f = fopen("contato.bin", "wb");
        if (f == NULL) {
            perror("contato.bin");
            return 1;
        }
        fwrite(&c, sizeof c, 1, f);
        fclose(f);
        printf("gravei %d %s (%zu bytes)\n", c.mat, c.nome, sizeof c);
        free(c.nome);
    } else {
        FILE *f = fopen("contato.bin", "rb");
        if (f == NULL) {
            perror("contato.bin");
            return 1;
        }
        fread(&c, sizeof c, 1, f);
        fclose(f);
        printf("li %d\n", c.mat);
        printf("nome: %s\n", c.nome);
    }
    return 0;
}


In [ ]:

!gcc -Wall -Wextra ponteiro.c -o ponteiro
!./ponteiro grava
!xxd contato.bin
!./ponteiro le


O arquivo tem 16 bytes: a matrícula, 4 bytes de preenchimento e os 8 bytes de um **endereço**. O nome
nunca foi para o disco. Na segunda execução, aquele endereço não pertence a ninguém, e o `printf` cai
(status -11 ou 139). Para gravar com `fwrite`, a `struct` precisa ser "plana": vetores de tamanho fixo,
como `char nome[50]`, e nenhum ponteiro.

Com `tam = 1`, o valor devolvido por `fread` é o número de **bytes** lidos — o jeito de copiar qualquer
arquivo, em blocos:

```c
char buf[4096];
size_t k;
while ((k = fread(buf, 1, sizeof buf, f)) > 0)
    fwrite(buf, 1, k, g);
```

### Exercício 2 — copiar qualquer arquivo

`./copia origem destino` copia o arquivo em blocos de 4096 bytes e imprime `<total> bytes copiados`.
Abra os dois arquivos em modo binário; se o segundo `fopen` falhar, feche o primeiro antes de sair com
status 1.


In [ ]:
%%writefile ex2.c
#include <stdio.h>

/* Exercicio 2: ./copia origem destino
   TODO: abra origem com "rb" e destino com "wb" (teste os dois NULL) e
   copie em blocos com fread/fwrite, somando os bytes copiados. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s origem destino\n", argv[0]);
        return 1;
    }
    long total = 0;

    /* TODO: implemente aqui */

    printf("%ld bytes copiados\n", total);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex2.c -o ex2
print('10000 bytes' + ':', end=' ')
testa('rm -f c1.bin; ./ex2 img.bin c1.bin && cmp img.bin c1.bin && echo iguais', '10000 bytes copiados\niguais')
print('arquivo vazio' + ':', end=' ')
testa('rm -f c2.bin; ./ex2 vazio.bin c2.bin && cmp vazio.bin c2.bin && echo iguais', '0 bytes copiados\niguais')
print('origem que não existe' + ':', end=' ')
testa('./ex2 nao_existe.bin c3.bin; echo "status $?"', 'status 1')


### Exercício 3 — comparar dois arquivos

`./compara a b` compara os arquivos byte a byte (bytes contados de zero) e imprime uma de três coisas:

```
iguais
diferem no byte 41
pref.bin termina antes, no byte 100
```

A última quando um arquivo é o começo do outro: o nome é o do arquivo mais curto, e o número é o seu
tamanho. Pode usar `fgetc` nos dois arquivos abertos com `"rb"` — lembre-se do `int c`.


In [ ]:
%%writefile ex3.c
#include <stdio.h>

/* Exercicio 3: ./compara a b
   TODO: leia um byte de cada arquivo por vez (fgetc) ate achar uma
   diferenca ou o fim de um deles. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s a b\n", argv[0]);
        return 1;
    }

    /* TODO: implemente aqui */

    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex3.c -o ex3
print('iguais' + ':', end=' ')
testa('./ex3 img.bin img.bin', 'iguais')
print('um byte trocado' + ':', end=' ')
testa('./ex3 img.bin img2.bin', 'diferem no byte 41')
print('prefixo primeiro' + ':', end=' ')
testa('./ex3 pref.bin img.bin', 'pref.bin termina antes, no byte 100')
print('prefixo segundo' + ':', end=' ')
testa('./ex3 img.bin pref.bin', 'pref.bin termina antes, no byte 100')
print('vazio' + ':', end=' ')
testa('./ex3 vazio.bin img.bin', 'vazio.bin termina antes, no byte 0')


## 4. Direto ao registro: `fseek` e `ftell`

Como todo registro tem `sizeof(struct aluno)` bytes, o registro *k* começa no byte
`k * sizeof(struct aluno)`, contando de zero, como num vetor.

```c
int  fseek(FILE *f, long desloc, int origem);   /* SEEK_SET, SEEK_CUR ou SEEK_END */
long ftell(FILE *f);                            /* posicao atual, em bytes */
void rewind(FILE *f);                           /* volta ao byte 0 */
```

`fseek(f, 0, SEEK_END)` seguido de `ftell(f)` dá o tamanho do arquivo. O deslocamento é `long` e pode
ser negativo; `sizeof` é `size_t`, sem sinal — converta para `long` antes de multiplicar ou negar.

Antes de executar: quem aparece com `k = 1`? E com `k = 4`, além do fim — o `fseek` falha?


In [ ]:
%%writefile consulta.c
/* consulta.c --- vai direto ao registro k de turma.bin
   uso: ./consulta turma.bin k */
#include <stdio.h>
#include <stdlib.h>
#include "aluno.h"

int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s arquivo k\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "rb");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }
    fseek(f, 0, SEEK_END);                       /* vai para o fim...  */
    long n = ftell(f) / (long) sizeof(struct aluno);  /* ...e mede */

    long k = atol(argv[2]);
    struct aluno a;
    fseek(f, k * (long) sizeof a, SEEK_SET);     /* salta k registros */
    if (fread(&a, sizeof a, 1, f) != 1) {
        printf("registro %ld nao existe (o arquivo tem %ld)\n", k, n);
        fclose(f);
        return 1;
    }
    printf("registro %ld de %ld: %d %s\n", k, n, a.mat, a.nome);
    fclose(f);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra consulta.c -o consulta
!./consulta turma.bin 1
!./consulta turma.bin 3
!./consulta turma.bin 4
!./consulta turma.bin -1


O registro 1 é o segundo aluno. Com `k = 4`, o `fseek` para o byte 352 **dá certo** — posicionar além do
fim é permitido; quem falha é o `fread`. Com `k = -1`, o próprio `fseek` falha e a posição fica onde
estava. Por isso o que se testa depois de um `fseek` é o retorno do `fread`.

### Exercício 4 — contar registros sem ler

`./registros arquivo` imprime `<n> registros de 88 bytes` usando só `fseek` e `ftell` — sem nenhum
`fread`. Use `sizeof(struct aluno)`, não o número 88. Se o tamanho não for múltiplo do registro, imprima
`arquivo corrompido: <r> bytes sobrando`.


In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include "aluno.h"

/* Exercicio 4: ./registros arquivo
   TODO: va ao fim com fseek, pegue o tamanho com ftell e divida por
   sizeof(struct aluno). */
int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "rb");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(f);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex4.c -o ex4
print('turma' + ':', end=' ')
testa('./ex4 turma_ex.bin', '4 registros de 88 bytes')
print('grande' + ':', end=' ')
testa('./ex4 grande.bin', '1000 registros de 88 bytes')
print('corrompido' + ':', end=' ')
testa('./ex4 corrompido.bin', 'arquivo corrompido: 10 bytes sobrando')
print('vazio' + ':', end=' ')
testa('./ex4 vazio.bin', '0 registros de 88 bytes')


### Exercício 5 — de trás para frente

`./inverte arquivo` lista os alunos do último ao primeiro, no formato `"%d %s\n"` (matrícula e nome),
com `fseek` + `fread` — sem guardar o arquivo na memória. Descubra o número de registros como no
exercício anterior.


In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include "aluno.h"

/* Exercicio 5: ./inverte arquivo
   TODO: descubra n e, para k de n-1 ate 0, salte ao registro k e leia. */
int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "rb");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(f);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex5.c -o ex5
print('turma' + ':', end=' ')
testa('./ex5 turma_ex.bin', '202600104 Davi Rocha\n202600103 Carla Menezes Santos\n202600102 Bruno Lima\n202600101 Ana Souza')
print('grande (5 primeiras linhas)' + ':', end=' ')
testa('./ex5 grande.bin | head -5', '202699956 Aluno 0999\n202699697 Aluno 0998\n202699688 Aluno 0997\n202699687 Aluno 0996\n202699651 Aluno 0995')
print('vazio' + ':', end=' ')
testa('./ex5 vazio.bin', '')


## 5. Alterar sem reescrever: `"r+b"`

`"r+b"` abre para ler **e** gravar, sem apagar o conteúdo. O `corrige.c` procura o aluno, troca a nota
na memória, recua um registro com `fseek(f, -(long) sizeof a, SEEK_CUR)` e grava por cima. Num fluxo
aberto com `+`, entre um `fread` e um `fwrite` é **obrigatório** um `fseek` (ou `rewind`); entre um
`fwrite` e um `fread`, um `fseek` ou `fflush`.


In [ ]:
%%writefile corrige.c
/* corrige.c --- troca uma nota dentro de turma.bin, sem reescrever o resto
   uso: ./corrige turma.bin matricula avaliacao nota */
#include <stdio.h>
#include <stdlib.h>
#include "aluno.h"

int main(int argc, char *argv[])
{
    if (argc != 5) {
        fprintf(stderr, "uso: %s arquivo matricula avaliacao nota\n", argv[0]);
        return 1;
    }
    int mat = atoi(argv[2]), av = atoi(argv[3]);
    double nota = atof(argv[4]);

    FILE *f = fopen(argv[1], "r+b");             /* ler e gravar */
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }
    struct aluno a;
    while (fread(&a, sizeof a, 1, f) == 1) {
        if (a.mat != mat)
            continue;
        a.v[av] = nota;
        fseek(f, -(long) sizeof a, SEEK_CUR);    /* volta um registro */
        if (fwrite(&a, sizeof a, 1, f) != 1)
            perror(argv[1]);
        else
            printf("%d: nota %d agora vale %.1f\n", mat, av, nota);
        fclose(f);
        return 0;
    }
    printf("matricula %d nao encontrada\n", mat);
    fclose(f);
    return 1;
}


In [ ]:

!gcc -Wall -Wextra corrige.c -o corrige
!./cadastro
!./corrige turma.bin 202600104 3 6.0
!./lista
!wc -c turma.bin


Agora a mesma coisa com o arquivo aberto em `"rb"` e sem testar o retorno do `fwrite`. Antes de
executar: a mensagem aparece? A média de Davi muda?


In [ ]:
%%writefile corrige_rb.c
/* corrige_rb.c --- o mesmo corrige.c, aberto com "rb"
   e sem testar o fwrite */
#include <stdio.h>
#include <stdlib.h>
#include "aluno.h"

int main(int argc, char *argv[])
{
    if (argc != 5) {
        fprintf(stderr, "uso: %s arquivo matricula avaliacao nota\n", argv[0]);
        return 1;
    }
    int mat = atoi(argv[2]), av = atoi(argv[3]);
    double nota = atof(argv[4]);

    FILE *f = fopen(argv[1], "rb");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }
    struct aluno a;
    while (fread(&a, sizeof a, 1, f) == 1) {
        if (a.mat != mat)
            continue;
        a.v[av] = nota;
        fseek(f, -(long) sizeof a, SEEK_CUR);
        fwrite(&a, sizeof a, 1, f);
        printf("%d: nota %d agora vale %.1f\n", mat, av, nota);
        break;
    }
    fclose(f);
    return 0;
}


In [ ]:

!gcc -Wall -Wextra corrige_rb.c -o corrige_rb
!./cadastro
!./corrige_rb turma.bin 202600104 3 6.0
!./lista


O programa anunciou a correção, mas a nota não mudou: o `fwrite` num fluxo aberto só para leitura falhou,
devolveu 0, e ninguém olhou. O compilador não tem como saber o modo — ele só é verificado em tempo de
execução. Teste o retorno de `fwrite` como você testa o de `fopen`.

### Exercício 6 — um `xxd` simples

`./hexa arquivo` mostra o conteúdo do arquivo, 16 bytes por linha, lidos com `fread(buf, 1, 16, f)`. Cada
linha tem:

- a posição do primeiro byte, com `"%08lx"`, seguida de **dois** espaços;
- cada byte com `"%02x "` (dois dígitos e um espaço); se a última linha tiver menos de 16 bytes, complete
  com três espaços por byte que falta;
- entre barras, cada byte como caractere se `isprint` for verdadeiro, ou `.` caso contrário.

```
00000000  4f 6c 61 2c 20 6d 75 6e 64 6f 21 0a 00 01 ff 7f |Ola, mundo!.....|
```


In [ ]:
%%writefile ex6.c
#include <stdio.h>
#include <ctype.h>

/* Exercicio 6: ./hexa arquivo
   TODO: leia blocos de 16 bytes com fread(buf, 1, 16, f) e imprima cada
   linha no formato do enunciado. Use unsigned char no buffer. */
int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "uso: %s arquivo\n", argv[0]);
        return 1;
    }
    FILE *f = fopen(argv[1], "rb");
    if (f == NULL) {
        perror(argv[1]);
        return 1;
    }

    /* TODO: implemente aqui */

    fclose(f);
    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra ex6.c -o ex6
print('ola.bin' + ':', end=' ')
testa('./ex6 ola.bin', '00000000  4f 6c 61 2c 20 6d 75 6e 64 6f 21 0a 00 01 ff 7f |Ola, mundo!.....|')
print('curto.bin (20 bytes)' + ':', end=' ')
testa('./ex6 curto.bin', '00000000  30 31 32 33 34 35 36 37 38 39 61 62 63 64 65 66 |0123456789abcdef|\n00000010  41 42 43 44                                     |ABCD|')
print('primeiro aluno' + ':', end=' ')
testa('./ex6 turma_ex.bin | head -6', '00000000  a5 6e 13 0c 41 6e 61 20 53 6f 75 7a 61 00 00 00 |.n..Ana Souza...|\n00000010  00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 |................|\n00000020  00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 |................|\n00000030  00 00 00 00 00 00 00 00 00 00 00 00 00 00 20 40 |.............. @|\n00000040  00 00 00 00 00 00 1a 40 00 00 00 00 00 00 22 40 |.......@......"@|\n00000050  00 00 00 00 00 00 1e 40 a6 6e 13 0c 42 72 75 6e |.......@.n..Brun|')
print('vazio' + ':', end=' ')
testa('./ex6 vazio.bin', '')


## Tarefas: altere o programa da aula

Volte às células do programa da aula, faça cada alteração, reexecute e confira se a saída bate com o que
você previu.

1. **Acrescentar alunos.** Troque o modo do `cadastro.c` para `"ab"` e rode-o duas vezes. Quantos alunos a
   `lista` mostra? E o que o `consulta` responde para `k = 5`? (Depois volte para `"wb"`.)
2. **Onde está a posição.** Na `lista`, imprima `ftell(f)` antes e depois de cada `fread`. Que números
   aparecem, e por que o primeiro é 0?
3. **Um cabeçalho.** Faça o `cadastro` gravar, antes dos alunos, um `int` com a quantidade: grave 0 no
   começo e, no fim, volte com `rewind` e grave o valor certo. Ajuste a `lista` e o `consulta` para
   pular esses 4 bytes — no `consulta`, o registro *k* passa a começar em `sizeof(int) + k * sizeof a`.

## Desafio: busca binária no disco

`grande.bin` tem 1000 alunos em ordem crescente de matrícula. `./busca arquivo mat` encontra o aluno
**sem ler o arquivo inteiro**: compare `mat` com o registro do meio, descarte a metade que não pode
contê-la e repita, sempre com `fseek` + `fread`. Use exatamente:

- `ini = 0`, `fim = n - 1` e, a cada passo, `meio = (ini + fim) / 2`;
- se a matrícula do meio for menor que `mat`, `ini = meio + 1`; se for maior, `fim = meio - 1`.

Conte quantos `fread` foram feitos e imprima `<mat> <nome> (leituras: <k>)` ou
`nao encontrado (leituras: <k>)`.


In [ ]:
%%writefile busca.c
#include <stdio.h>
#include <stdlib.h>
#include "aluno.h"

/* Desafio: ./busca arquivo matricula
   TODO: busca binaria com fseek + fread, contando as leituras. */
int main(int argc, char *argv[])
{
    if (argc != 3) {
        fprintf(stderr, "uso: %s arquivo matricula\n", argv[0]);
        return 1;
    }

    /* TODO: sua solucao aqui */

    return 0;
}


In [ ]:
!gcc -g -Wall -Wextra busca.c -o busca
print('Carla' + ':', end=' ')
testa('./busca turma_ex.bin 202600103', '202600103 Carla Menezes Santos (leituras: 2)')
print('inexistente na turma' + ':', end=' ')
testa('./busca turma_ex.bin 202600999', 'nao encontrado (leituras: 3)')
print('primeiro' + ':', end=' ')
testa('./busca grande.bin 202600041', '202600041 Aluno 0000 (leituras: 9)')
print('último' + ':', end=' ')
testa('./busca grande.bin 202699956', '202699956 Aluno 0999 (leituras: 10)')
print('do meio' + ':', end=' ')
testa('./busca grande.bin 202662238', '202662238 Aluno 0617 (leituras: 10)')
print('menor que todos' + ':', end=' ')
testa('./busca grande.bin 202500000', 'nao encontrado (leituras: 9)')
print('maior que todos' + ':', end=' ')
testa('./busca grande.bin 202800000', 'nao encontrado (leituras: 10)')
print('entre dois' + ':', end=' ')
testa('./busca grande.bin 202630990', 'nao encontrado (leituras: 10)')


Depois de passar no teste: com um milhão de alunos, quantas leituras a busca faz, no máximo? E uma
busca que lê do início até achar? É por isso que bancos de dados guardam registros de tamanho fixo e
índices ordenados.

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão em:

- King (2008), *C Programming: A Modern Approach*, §22.1 — fluxos texto e binários; §22.2 — modos;
  §22.6 — `fread` e `fwrite`; §22.7 — `fseek`, `ftell` e `rewind`;
- Kernighan e Ritchie (1988), *The C Programming Language*, §6.4 — ponteiros para estruturas; §8.4 —
  acesso aleatório; Apêndice B1.5–B1.6 — `fread`, `fwrite`, `fseek`, `ftell`;
- Deitel e Deitel (2011), *C: como programar*, cap. 11 — arquivos de acesso aleatório;
- ISO/IEC 9899:2018 — §6.7.2.1 (preenchimento em estruturas), §7.21.2 (fluxos binários), §7.21.5.3
  (modo de atualização), §7.21.8 (`fread`, `fwrite`) e §7.21.9 (`fseek`, `ftell`);
- Bryant e O'Hallaron (2016), *Computer Systems: A Programmer's Perspective*, §2.1.3 — ordem dos bytes;
  §3.9.3 — alinhamento;
- Goldberg (1991), *What Every Computer Scientist Should Know About Floating-Point Arithmetic* —
  representação de `double`;
- Kerrisk (2010), *The Linux Programming Interface*, §4.7 — posicionar além do fim do arquivo;
- Knuth (1998), *The Art of Computer Programming*, vol. 3, §6.2.1 — busca binária (Algoritmo B).
